In [ ]:
# ==============================================================================
# 1. AUTOMATYCZNE ODŚWIEŻANIE MODUŁÓW (BEZPIECZNE DLA PYTHON 3.12+)
# ==============================================================================
try:
    # Standardowe ładowanie, jeśli IPython jest zaktualizowany
    %load_ext autoreload
    %autoreload 2
except Exception:
    # Alternatywne obejście, jeśli środowisko zgłasza brak modułu 'imp'
    import ipykernel
    print("⚠️ Standard autoreload failed, applying compatibility patch for Python 3.12+...")
    # W nowym Pythonie używa się importlib zamiast imp
    import sys
    import types
    from importlib import reload
    # Wstrzykujemy 'imp' do sys.modules w locie, żeby oszukać stare skrypty IPythona
    imp_mock = types.ModuleType('imp')
    imp_mock.reload = reload
    sys.modules['imp'] = imp_mock
    
    # Próbujemy załadować ponownie
    %load_ext autoreload
    %autoreload 2

import os
import sys

# ==============================================================================
# 2. DEFINICJA REPOZYTORIUM GITHUB (DLA SESJI W CHMURZE COLABA)
# ==============================================================================
GITHUB_USER = "rgorocki" 
REPO_NAME = "projekt_analiza_obrazu"
REPO_URL = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"

# Sprawdzanie: chmura Google Colab czy lokalny workspace w VS Code
IS_COLAB_CLOUD = os.path.exists("/content")

if IS_COLAB_CLOUD:
    print("🚀 [STATUS] Detected Google Colab Cloud Environment.")
    # Klonowanie repozytorium lub pobieranie najnowszych zmian (git pull)
    if not os.path.exists(REPO_NAME):
        print(f"-> Cloning repository: {REPO_NAME}...")
        !git clone {REPO_URL}
    else:
        print(f"-> Repository already exists. Pulling latest updates from GitHub...")
        !cd {REPO_NAME} && git stash && git pull
        
    # Dodanie folderu projektu do ścieżki wyszukiwania Pythona w chmurze
    path_to_append = os.path.abspath(REPO_NAME)
    if path_to_append not in sys.path:
        sys.path.append(path_to_append)
else:
    print("💻 [STATUS] Detected Local VS Code Environment.")
    print("-> Using local files from your workspace. Skipping git clone/pull.")
    
    # W VS Code dodajemy katalog nadrzędny workspace'u, aby widzieć moduły 'src'
    current_dir = os.getcwd()
    parent_dir = os.path.dirname(current_dir)
    if parent_dir not in sys.path:
        sys.path.append(parent_dir)

print("\n✅ Environment configured successfully! Ready to run the pipeline.")

In [ ]:
import cv2
import os
import glob
import matplotlib.pyplot as plt
import pandas as pd

# Import dedykowanych klas modułowych z folderu src
from src.wcag_analyzer import WCAGAnalyzer
from src.segmenter import KMeansSegmenter
from src.detector import UIDetector
from src.saliency_mapper import SaliencyMapper

# ==============================================================================
# 1. DYNAMICZNE MAPOWANIE KATALOGÓW WEJŚCIOWYCH I WYJŚCIOWYCH
# ==============================================================================
if IS_COLAB_CLOUD:
    dir_screenshots = f"{REPO_NAME}/data/screenshots"
    dir_icons = f"{REPO_NAME}/data/icons"
    dir_layouts = f"{REPO_NAME}/data/layouts"
    dir_output = f"{REPO_NAME}/output"
else:
    dir_screenshots = "../data/screenshots"
    dir_icons = "../data/icons"
    dir_layouts = "../data/layouts"
    dir_output = "../output"

# Tworzenie katalogu na wyniki
os.makedirs(dir_output, exist_ok=True)

# Funkcja pomocnicza do pobierania obrazów z rozszerzeniami PNG/JPG
def get_image_paths(folder_path):
    extensions = ["*.png", "*.jpg", "*.jpeg"]
    paths = []
    for ext in extensions:
        paths.extend(glob.glob(os.path.join(folder_path, ext)))
    return paths

# Pobieramy listy plików dla poszczególnych zadań
screenshot_files = get_image_paths(dir_screenshots)
icon_files = get_image_paths(dir_icons)
layout_files = get_image_paths(dir_layouts)

print(f"📊 Dataset Stats:")
print(f" |- Screenshots Found: {len(screenshot_files)}")
print(f" |- Icons/UI Elements Found: {len(icon_files)}")
print(f" |- Figma Layouts Found: {len(layout_files)}\n")

# Lista do zbierania rekordów statystycznych do końcowego raportu CSV
batch_report_data = []

# ==============================================================================
# 2. SEKWENCYJNE PRZETWARZANIE SYSTEMOWE (PĘTLA GŁÓWNA)
# ==============================================================================
print("====== STARTING BATCH PROCESSING ======")

# --- ETAP 1: Analiza interfejsów (Screenshots) -> Detekcja, Kontrast, Segmentacja ---
for idx, path in enumerate(screenshot_files):
    filename = os.path.basename(path)
    print(f"[{idx+1}/{len(screenshot_files)}] Processing screenshot: {filename}")
    
    try:
        # Inicjalizacja komponentów
        detector = UIDetector(min_area=250)
        analyzer = WCAGAnalyzer()
        segmenter = KMeansSegmenter(k_clusters=5)
        
        # Wykonanie algorytmów CV
        annotated_img, bboxes = detector.detect_elements(path)
        segmented_img = segmenter.segment(path)
        
        # Przykładowa analiza kontrastu WCAG (para kolorów tło-tekst)
        # W pełnej wersji wartości te można ładować dynamicznie z pliku wcag_colors.json
        contrast_ratio = analyzer.get_contrast_ratio((255, 255, 255), (30, 30, 30))
        wcag_status = analyzer.evaluate_accessibility(contrast_ratio)
        
        # Zapis grafik wynikowych do folderu output
        name_clean = os.path.splitext(filename)[0]
        cv2.imwrite(os.path.join(dir_output, f"{name_clean}_detected.png"), annotated_img)
        cv2.imwrite(os.path.join(dir_output, f"{name_clean}_segmented.png"), segmented_img)
        
        # Zapis danych do zbiorczego zestawienia
        batch_report_data.append({
            "Dataset_Type": "Screenshot",
            "File_Name": filename,
            "Detected_Elements": len(bboxes),
            "WCAG_Contrast_Ratio": round(contrast_ratio, 2),
            "WCAG_AA_Pass": wcag_status["AA_normal"]
        })
    except Exception as e:
        print(f" ❌ Error processing {filename}: {e}")

# --- ETAP 2: Generowanie map percepcji użytkownika (Figma Layouts) ---
for idx, path in enumerate(layout_files):
    filename = os.path.basename(path)
    print(f"[{idx+1}/{len(layout_files)}] Generating Saliency Map for layout: {filename}")
    
    try:
        mapper = SaliencyMapper()
        saliency_map, heatmap_overlay = mapper.generate_map(path)
        
        name_clean = os.path.splitext(filename)[0]
        cv2.imwrite(os.path.join(dir_output, f"{name_clean}_saliency_heatmap.png"), heatmap_overlay)
        
        batch_report_data.append({
            "Dataset_Type": "Figma_Layout",
            "File_Name": filename,
            "Detected_Elements": "N/A",
            "WCAG_Contrast_Ratio": "N/A",
            "WCAG_AA_Pass": "N/A"
        })
    except Exception as e:
        print(f" ❌ Error generating saliency for {filename}: {e}")

# ==============================================================================
# 3. ZAPIS RAPORTU KOŃCOWEGO I WIZUALIZACJA PODSUMOWANIA
# ==============================================================================
print("\n====== PROCESSING COMPLETED ======")

if batch_report_data:
    # Tworzenie obiektu DataFrame i zapis do pliku CSV
    df_final_report = pd.DataFrame(batch_report_data)
    csv_output_path = os.path.join(dir_output, "ux_cv_system_report.csv")
    df_final_report.to_csv(csv_output_path, index=False)
    print(f"💾 Comprehensive UX report saved successfully to: {csv_output_path}\n")
    
    # Wyświetlenie wygenerowanej tabeli podsumowującej bezpośrodnio w notesie
    display(df_final_report)
else:
    print("⚠️ No data was processed. Please verify that your data/ folders contain images.")